# Clusterwise self-vs-other GLM + global-most-populated-cluster normalization

Combines two scripts into one notebook, run top to bottom:

1. **`scripts/cluster_glm_reliability.py`** -- per-semantic-cluster self-vs-other
   beta cosine distance, using the LLaMA-pipeline data loading from
   `scripts/semantic_glm.py` and the validated sklearn Poisson-ridge fitter from
   `neural_encoding/reliability.py` (`fit_poisson_ridge_beta_sklearn` +
   `select_alpha_cv` -- the same fitter `run_beta_reliability_all_neurons` uses
   for the validated self/other `r_cross` comparison) instead of the ad hoc
   `PoissonRegressor`+`GridSearchCV` fitter the old BERT-pipeline notebook used.

   Cluster labels come from the canonical
   `Transcripts/{pid}_filtered_used_rows_withNP_withClusterIDNew.xlsx`
   (`FinalClusterID` column), which is also the source the embedding-cache
   builder (`p6/run_all_patients_multi_model.py`) reads before writing
   `{pid}_llama-3.1-8b_ctx200_word_emb_layers.npy`. That builder applies
   `dropna(onset).sort_values(onset).reset_index` before writing the cache, so
   cluster IDs go through the identical transform here to land on the correct
   row -- guarded by a hard row-count check against the cache shape.

2. **`scripts/normalize_clusterwise_global_most_populated.py`** -- normalizes
   each cluster's raw cosine distance by a noise floor derived from whichever
   semantic cluster has the most trials *summed across all patients* (ported
   from `resultsofsemantic.ipynb`'s `normalization_mode="global_most_populated"`).


## Setup

In [ ]:
import os
import sys
import warnings
import importlib.util

# Must run before numpy/sklearn import. With n_jobs worker processes each
# also spawning a multi-threaded BLAS call underneath, a many-core box lets
# every worker try to grab dozens of cores for matrix ops far too small to
# benefit -- pure contention. The actual parallelism comes from the outer
# joblib processes (one PoissonRegressor fit per worker), not from BLAS.
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")

import numpy as np
import pandas as pd
from scipy.special import gammaln
from scipy.stats import f_oneway
from sklearn.decomposition import PCA
from sklearn.linear_model import PoissonRegressor
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

In [ ]:
# ── paths ─────────────────────────────────────────────────────────────────────
PROJECT_ROOT    = "/scratch/aniluchavez/hippocampal-speaker-semantics"
EMBED_DIR       = "/scratch/aniluchavez/ConvoDATAS/EmbedCache"
SPIKE_ROOT      = "/scratch/aniluchavez/ConvoDATAS/SpikeWindows"
TRANSCRIPT_ROOT = "/scratch/aniluchavez/ConvoDATAS/Transcripts"
RESULTS_ROOT    = "/scratch/aniluchavez/ConvoDATAS/SemanticGLM/clusterwise"

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

## Load `neural_encoding/cluster_analysis.py`

Loaded directly by path rather than `from neural_encoding.cluster_analysis import ...`,
because `neural_encoding/__init__.py` eagerly imports `regression.py`, which is
broken under this env's Python 3.12 (a dataclass field has a mutable numpy-array
default, which `dataclasses` now rejects). Matches the convention
`scripts/semantic_glm.py` and `scripts/run_reliability.py` already use for the
same reason.

In [ ]:
def _load_module(rel_path, name):
    path = os.path.join(PROJECT_ROOT, rel_path)
    if name in sys.modules:
        del sys.modules[name]
    spec = importlib.util.spec_from_file_location(name, path)
    mod = importlib.util.module_from_spec(spec)
    sys.modules[name] = mod
    spec.loader.exec_module(mod)
    return mod


_cluster_mod = _load_module("neural_encoding/cluster_analysis.py", "nn_cluster_analysis")

report_cluster_balance = _cluster_mod.report_cluster_balance
run_clusterwise_cosine_distance_bootstrap = _cluster_mod.run_clusterwise_cosine_distance_bootstrap

## Validated GLM fitter

`fit_poisson_ridge_beta_sklearn` / `select_alpha_cv` / `poisson_ll_numpy`,
copied verbatim from `neural_encoding/reliability.py` rather than imported --
`run_clusterwise_cosine_distance`'s `run_poisson_ridge` callback runs inside
`joblib.Parallel`, and a process-pool worker can only unpickle a function by
reference if it's a normal top-level function in a *really* importable
module. Defining them directly here means they live in this notebook's
`__main__` namespace, which `cloudpickle` (joblib's pickler) special-cases to
serialize by value -- this is the standard, well-supported way
`joblib.Parallel` works from interactively-defined notebook functions.

In [ ]:
def poisson_ll_numpy(y_true, mu_pred):
    y_true = np.asarray(y_true, dtype=float)
    mu_pred = np.clip(np.asarray(mu_pred, dtype=float), 1e-10, None)
    return float(np.sum(y_true * np.log(mu_pred) - mu_pred - gammaln(y_true + 1)))


def fit_poisson_ridge_beta_sklearn(X, y, alpha, *, fit_intercept=True, max_iter=1000):
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    model = PoissonRegressor(alpha=float(alpha), fit_intercept=fit_intercept, max_iter=max_iter)
    model.fit(X, y)
    return model.coef_.astype(float, copy=True)


def select_alpha_cv(X, y, alphas, *, n_splits=5, random_state=0, standardize=True):
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    n = X.shape[0]
    if n < max(10, 2 * n_splits):
        return float(alphas[0])

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    splits_list = list(kf.split(X, y))

    best_alpha = float(alphas[0])
    best_score = -np.inf
    for a in alphas:
        ll_list = []
        for tr, va in splits_list:
            Xtr, Xva = X[tr], X[va]
            ytr, yva = y[tr], y[va]
            if standardize:
                sc = StandardScaler(with_mean=True, with_std=True)
                Xtr_s = sc.fit_transform(Xtr)
                Xva_s = sc.transform(Xva)
            else:
                Xtr_s, Xva_s = Xtr, Xva
            m = PoissonRegressor(alpha=float(a), fit_intercept=True, max_iter=1000)
            m.fit(Xtr_s, ytr)
            mu_va = m.predict(Xva_s)
            ll_list.append(poisson_ll_numpy(yva, mu_va))
        score = float(np.mean(ll_list)) if ll_list else -np.inf
        if score > best_score:
            best_score = score
            best_alpha = float(a)
    return best_alpha

## Config

In [ ]:
# ── GLM config (matches scripts/semantic_glm.py defaults) ─────────────────────
MODEL_TAG    = "llama-3.1-8b"
CONTEXT_TAG  = "_ctx200"  # matches what's actually on disk in EmbedCache --
                          # no patient has a bare (no-context-tag) llama-3.1-8b
                          # cache file; all are *_ctx200_word_emb_layers.npy
LAYER        = 18
N_COMPONENTS = 100
MIN_SPIKES   = 5
ALPHAS       = np.logspace(-3, 3, 30)
WINDOW_TAG   = "tshift-150_tlen500_oshift+200_olen500"   # "fixed" window convention
N_BOOTSTRAP          = 20  # resampled-fit repeats for the larger side per cluster
N_HALFSPLIT_REPEATS  = 5   # repeated 50/50 splits per side for reliability

# ── batch-run config (CLI flags in the script become plain variables here) ────
RUN_PATIENT = None   # set to a patient_ID string to run just one, else None for all
RUN_REGION  = None   # set to a region name to run just one, else None for all
N_JOBS      = 16

# ── patients (verbatim from scripts/semantic_glm.py) ───────────────────────────
PATIENTS = [
    {"patient_ID": "PTYEU_task147", "patient": "ptYEU_task147",
     "region_ranges": {"hippocampus": [(1,16),(25,40)], "ACC": [(17,24),(41,48)]}},
    {"patient_ID": "PTYFF_task17",  "patient": "ptYFF_task17",
     "region_ranges": {"hippocampus": [(9,16),(25,40)], "ACC": [(17,24),(41,48)]}},
    {"patient_ID": "PTYFG_task18",  "patient": "ptYFG_task18",
     "region_ranges": {"hippocampus": [(9,16)],         "ACC": [(25,56)]}},
    {"patient_ID": "PTYFI_task81",  "patient": "ptYFI_task81",
     "region_ranges": {"hippocampus": [(1,8),(25,40)],  "ACC": [(9,16)]}},
    {"patient_ID": "PTYFA_task25",  "patient": "ptYFA_task25",
     "region_ranges": {"hippocampus": [(1,16),(25,40)], "ACC": [(17,24)]}},
    {"patient_ID": "PTYFK_task40",  "patient": "ptYFK_task40",
     "region_ranges": {"hippocampus": [(1,16),(25,40)], "ACC": [(49,56)]}},
    {"patient_ID": "PTYEY_task86",  "patient": "ptYEY_task86",
     "region_ranges": {"hippocampus": [(1,16)]}},
    {"patient_ID": "PTYEV_task37",  "patient": "ptYEV_task37",
     "region_ranges": {"hippocampus": [(1,16),(25,40)], "ACC": [(17,24),(41,48)]}},
    {"patient_ID": "PTYEZ_task60",  "patient": "ptYEZ_task60",
     "region_ranges": {"hippocampus": [(1,16)],         "ACC": [(17,24)]}},
    {"patient_ID": "PTYFC_task28",  "patient": "ptYFC_task28",
     "region_ranges": {"hippocampus": [(1,8),(33,48)],  "ACC": [(17,32),(49,64)]}},
    {"patient_ID": "PTYFM_task104", "patient": "ptYFM_task104",
     "region_ranges": {"hippocampus": [(33,48)]}},
    {"patient_ID": "PTYFP_task88",  "patient": "ptYFP_task88",
     "region_ranges": {"hippocampus": [(17,24),(25,32),(49,56),(57,64)]}},
    {"patient_ID": "PTYFR_task91",  "patient": "ptYFR_task91",
     "region_ranges": {"hippocampus": [(1,16),(41,56)]}},
    {"patient_ID": "PTYFS_task95",  "patient": "ptYFS_task95",
     "region_ranges": {"hippocampus": [(1,24)]}},
    {"patient_ID": "PTYFU_task224", "patient": "ptYFU_task224",
     "region_ranges": {"hippocampus": [(17,32),(41,56)]}},
]

## Data-loading helpers

Copied verbatim from `scripts/semantic_glm.py` -- that module runs its full
analysis loop at import time (no `__main__` guard), so it can't be imported
directly.

In [ ]:
def load_spike_matrix(spike_dir, speaker, region):
    spk_dir = os.path.join(spike_dir, speaker)
    if not os.path.isdir(spk_dir):
        return None
    cands = [f for f in os.listdir(spk_dir)
             if f.lower().startswith(region.lower()) and f.endswith("_spike_counts.npy")]
    return np.load(os.path.join(spk_dir, cands[0])) if cands else None


def load_speaker_assignment(spike_dir):
    cands = [f for f in os.listdir(spike_dir) if f.endswith("_with_regress_dur.xlsx")]
    if not cands:
        raise FileNotFoundError(f"No _with_regress_dur.xlsx in {spike_dir}")
    tx = pd.read_excel(os.path.join(spike_dir, cands[0]))
    spk_cols = sorted(
        [c for c in tx.columns if str(c).startswith("Speaker")],
        key=lambda c: int(c.replace("Speaker", "").strip())
                      if c.replace("Speaker", "").strip().isdigit() else 999,
    )

    def _nn(val):
        return pd.notna(val) and str(val).strip() not in ("", "nan")

    dir_membership = {
        col: np.array([_nn(v) for v in tx[col]], dtype=bool) for col in spk_cols
    }
    n = len(tx)
    assign = np.array([None] * n, dtype=object)
    for i in range(n):
        for col in spk_cols:
            if dir_membership[col][i]:
                assign[i] = col
                break
    mask_self = assign == "Speaker1"
    mask_other = np.array([(a is not None and a != "Speaker1") for a in assign], dtype=bool)
    return assign, mask_self, mask_other, dir_membership


def load_condition_ordered(spike_dir, cond, region, spk_assignment, dir_membership):
    if cond == "self":
        return load_spike_matrix(spike_dir, "Speaker1", region)
    other_spks = sorted([c for c in dir_membership if c != "Speaker1"])
    mats = {s: load_spike_matrix(spike_dir, s, region) for s in other_spks}
    mats = {s: m for s, m in mats.items() if m is not None}
    if not mats:
        return None
    dir_pos = {s: 0 for s in mats}
    rows = []
    for i, spk in enumerate(spk_assignment):
        if spk is None or spk == "Speaker1":
            continue
        for s in mats:
            if dir_membership[s][i]:
                if spk == s:
                    rows.append(mats[s][dir_pos[s]])
                dir_pos[s] += 1
    return np.vstack(rows) if rows else None


def find_spike_dir(patient):
    d = os.path.join(SPIKE_ROOT, f"output_{patient}_english_only_{WINDOW_TAG}")
    return d if os.path.isdir(d) else None


def load_cluster_ids(patient_ID, n_words_expected):
    """FinalClusterID from the canonical Transcripts xlsx, aligned to the
    embedding-cache row order via the same to_numeric/dropna/sort transform
    extract_embeddings_ctx200.py applies before writing the cache, then
    hard-checked against the cache's row count.

    Some patients' "New" filename is a stale/broken symlink (target since
    deleted); extract_control_features.py already tolerates a "Newest"
    suffix variant via regex, so check both filenames here too."""
    path = os.path.join(
        TRANSCRIPT_ROOT, f"{patient_ID}_filtered_used_rows_withNP_withClusterIDNew.xlsx")
    if not os.path.exists(path):
        alt_path = os.path.join(
            TRANSCRIPT_ROOT, f"{patient_ID}_filtered_used_rows_withNP_withClusterIDNewest.xlsx")
        if os.path.exists(alt_path):
            path = alt_path
        else:
            print(f"  {patient_ID}: no transcript/cluster file at {path} (or Newest variant)", flush=True)
            return None
    df = pd.read_excel(path)
    if "FinalClusterID" not in df.columns or "onset" not in df.columns:
        print(f"  {patient_ID}: missing FinalClusterID/onset column in {path}", flush=True)
        return None
    df["onset"] = pd.to_numeric(df["onset"], errors="coerce")
    df = df.dropna(subset=["onset"]).sort_values("onset").reset_index(drop=True)
    if len(df) != n_words_expected:
        print(f"  [WARN] {patient_ID}: cluster-ID rows ({len(df)}) != "
              f"embedding-cache rows ({n_words_expected}) -- skipping cluster IDs", flush=True)
        return None
    return df["FinalClusterID"].values

## Per-patient/region data builder

In [ ]:
def build_patient_region_data(cfg, region, layer, n_components):
    patient_ID = cfg["patient_ID"]
    patient = cfg["patient"]

    npy_path = os.path.join(EMBED_DIR, f"{patient_ID}_{MODEL_TAG}{CONTEXT_TAG}_word_emb_layers.npy")
    spike_dir = find_spike_dir(patient)
    if not os.path.exists(npy_path) or spike_dir is None:
        print(f"  {patient_ID}: missing embeddings or spike dir -- skip", flush=True)
        return None

    try:
        spk_assignment, mask_self, mask_other, dir_membership = load_speaker_assignment(spike_dir)
    except FileNotFoundError as e:
        print(f"  {patient_ID}: {e} -- skip", flush=True)
        return None

    X_layer_raw = np.load(npy_path, mmap_mode="r")[layer].astype(np.float32)

    cluster_ids_full = load_cluster_ids(patient_ID, X_layer_raw.shape[0])
    if cluster_ids_full is None:
        return None

    cond_data = {}
    for cond, mask in [("self", mask_self), ("other", mask_other)]:
        Y_mat = load_condition_ordered(spike_dir, cond, region, spk_assignment, dir_membership)
        if Y_mat is None or Y_mat.ndim < 2:
            cond_data[cond] = None
            continue
        if Y_mat.shape[0] != int(mask.sum()):
            print(f"  {region}/{cond}: row mismatch -- skip", flush=True)
            cond_data[cond] = None
            continue
        valid = ~np.isnan(Y_mat).any(axis=1)
        cond_data[cond] = (Y_mat[valid].astype(np.float32), mask, valid)

    if cond_data.get("self") is None or cond_data.get("other") is None:
        print(f"  {patient_ID}/{region}: missing self or other data -- skip", flush=True)
        return None

    Y_self_raw, mask_s, valid_s = cond_data["self"]
    Y_other_raw, mask_o, valid_o = cond_data["other"]

    spike_ok = (Y_self_raw.sum(0) >= MIN_SPIKES) & (Y_other_raw.sum(0) >= MIN_SPIKES)
    if spike_ok.sum() == 0:
        print(f"  {patient_ID}/{region}: 0 neurons pass spike filter -- skip", flush=True)
        return None

    # Joint per-patient PCA on the full word set, matching semantic_glm.py's
    # --reliability branch (PCA fit once, self/other split out afterward).
    X_layer_pca = PCA(n_components=n_components).fit_transform(
        np.asarray(X_layer_raw, dtype=np.float64))

    X_self = StandardScaler().fit_transform(X_layer_pca[mask_s][valid_s])
    X_other = StandardScaler().fit_transform(X_layer_pca[mask_o][valid_o])
    Y_self = Y_self_raw[:, spike_ok].astype(np.float64)
    Y_other = Y_other_raw[:, spike_ok].astype(np.float64)

    cluster_self = cluster_ids_full[mask_s][valid_s]
    cluster_other = cluster_ids_full[mask_o][valid_o]

    print(f"  {patient_ID}/{region}: self={X_self.shape} other={X_other.shape} "
          f"neurons={int(spike_ok.sum())}", flush=True)

    return dict(
        X_self=X_self, X_other=X_other, Y_self=Y_self, Y_other=Y_other,
        metadata_self=pd.DataFrame({"ClusterID": cluster_self}),
        metadata_other=pd.DataFrame({"ClusterID": cluster_other}),
        mask_self=mask_s, valid_self=valid_s,
        mask_other=mask_o, valid_other=valid_o,
    )

## Beta-fit adapter

The actual "swap in the updated GLM" step: wraps the validated
`select_alpha_cv` + `fit_poisson_ridge_beta_sklearn` fitter to match the
`run_poisson_ridge(X, Y, ..., fast_beta_only=True)` signature
`run_clusterwise_cosine_distance` expects.

In [ ]:
def make_beta_fit_adapter(alphas):
    def _adapter(X, Y, patient_id, neuron_idx, region_name, n_semantic_dims,
                 results_dir, fast_beta_only=True, save_results=False):
        y = np.asarray(Y)[:, neuron_idx]
        alpha = select_alpha_cv(X, y, alphas)
        coef = fit_poisson_ridge_beta_sklearn(X, y, alpha)
        return pd.DataFrame([coef], columns=[f"beta_{i}" for i in range(len(coef))])
    return _adapter


run_poisson_ridge = make_beta_fit_adapter(ALPHAS)

## Run the batch

Per patient x region: for each semantic cluster, fit self/other betas and
compute their cosine distance per cluster/neuron, using each cluster's own
natural `min(n_self, n_other)` rather than a forced cross-cluster median
(`run_clusterwise_cosine_distance_bootstrap`, in `neural_encoding/cluster_analysis.py`).
A cluster with far more self than other trials (e.g. function words) still
only fits `min(self, other)` trials per side at a time, but the larger side
gets re-sampled and re-fit `N_BOOTSTRAP` times and the cosine distance is
averaged across draws -- using far more of that side's data over the run than
one arbitrary fixed downsample would, instead of discarding it down to a
tiny shared median. `minimal_balancing`'s pre-fit global downsampling is no
longer used here for the same reason (it also discarded data unnecessarily;
`report_cluster_balance` below is kept purely as an informational diagnostic).

Default (process-pool/`loky`) joblib backend: `run_poisson_ridge`
and the functions it calls are plain top-level functions in this notebook's
`__main__` namespace, so `cloudpickle` can ship them to worker processes by
value -- real multi-core parallelism instead of GIL-bound threads, with each
worker's own BLAS calls kept single-threaded (env vars above) so `N_JOBS`
workers don't oversubscribe the box.

In [ ]:
patients = [p for p in PATIENTS if RUN_PATIENT is None or p["patient_ID"] == RUN_PATIENT]

for cfg in patients:
    patient_ID = cfg["patient_ID"]
    regions = [r for r in cfg["region_ranges"] if RUN_REGION is None or r == RUN_REGION]

    for region in regions:
        print(f"\n{'='*60}\n  {patient_ID} / {region}", flush=True)

        data = build_patient_region_data(cfg, region, LAYER, N_COMPONENTS)
        if data is None:
            continue

        # Diagnostic only -- not used to pre-trim the data (see markdown above).
        report_cluster_balance(
            data["metadata_self"], data["metadata_other"], patient_ID, region)

        run_clusterwise_cosine_distance_bootstrap(
            X_self=data["X_self"], X_other=data["X_other"],
            Y_self=data["Y_self"], Y_other=data["Y_other"],
            metadata_self=data["metadata_self"], metadata_other=data["metadata_other"],
            cluster_column="ClusterID",
            region_name=region, patient_id=patient_ID,
            n_components=N_COMPONENTS,
            results_root=RESULTS_ROOT,
            run_poisson_ridge=run_poisson_ridge,
            min_trials_per_condition=10,
            n_bootstrap=N_BOOTSTRAP,
            n_halfsplit_repeats=N_HALFSPLIT_REPEATS,
            compute_half_splits=True,
            n_jobs=N_JOBS,
            print_trial_counts=True,
        )

print("\nDone.", flush=True)

## Normalization: global most-populated cluster

Ported from `resultsofsemantic.ipynb`'s `normalization_mode="global_most_populated"`
(cell 8): find the semantic cluster with the most halfsplit-valid trials
*summed across all patients* per region, use that cluster's half-split
reliability as a noise floor per patient
(`1 - min(self_halfsplit_mean, other_halfsplit_mean)`), and divide every
cluster's raw cosine distance by it. Reads `cosine_distance` directly from
the CSVs `run_clusterwise_cosine_distance` just saved (no need to recompute
it from the raw-betas pickle, unlike the original notebook cell).

In [ ]:
HALFSPLIT_COLS = ["self_halfsplit_cosine", "other_halfsplit_cosine"]
NORM_REGIONS = ["hippocampus", "ACC"]


def find_global_majority_cluster(patient_dirs, region):
    """Cluster_id with the most halfsplit-valid trials, summed across all
    patients, for this region."""
    cluster_counts = {}
    for pdir in patient_dirs:
        csv_path = os.path.join(pdir, f"{region}_clusterwise_cosine_distances.csv")
        if not os.path.exists(csv_path):
            continue
        df = pd.read_csv(csv_path)
        if not set(HALFSPLIT_COLS).issubset(df.columns):
            continue
        df = df.dropna(subset=HALFSPLIT_COLS)
        for k, v in df["cluster_id"].value_counts().items():
            cluster_counts[k] = cluster_counts.get(k, 0) + v
    if not cluster_counts:
        return None
    return max(cluster_counts, key=cluster_counts.get)


def normalize_patient_region(csv_path, global_majority_cluster, region):
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()

    if not set(HALFSPLIT_COLS).issubset(df.columns):
        print(f"  {csv_path}: missing halfsplit columns "
              f"(re-run the batch cell above with compute_half_splits=True) -- skip")
        return None

    df_valid = df.dropna(subset=HALFSPLIT_COLS)
    ref_df = df_valid[df_valid["cluster_id"] == global_majority_cluster]

    noise_floor = np.nan
    if not ref_df.empty:
        self_mean = ref_df["self_halfsplit_cosine"].mean(skipna=True)
        other_mean = ref_df["other_halfsplit_cosine"].mean(skipna=True)
        if np.isfinite(self_mean) or np.isfinite(other_mean):
            noise_floor = 1 - np.nanmin([self_mean, other_mean])

    if np.isfinite(noise_floor) and noise_floor > 0:
        df["normalized_distance"] = df["cosine_distance"] / noise_floor
    else:
        df["normalized_distance"] = np.nan

    df["reference_cluster"] = global_majority_cluster
    df["noise_floor"] = noise_floor

    n_valid = df["normalized_distance"].notna().sum()
    patient_id = os.path.basename(os.path.dirname(csv_path))
    if np.isfinite(noise_floor):
        print(f"  {patient_id}/{region}: ref_cluster={global_majority_cluster} "
              f"noise_floor={noise_floor:.4f} ({n_valid}/{len(df)} rows normalized)")
    else:
        print(f"  {patient_id}/{region}: ref_cluster={global_majority_cluster} "
              f"has no usable halfsplit data -- not normalized")

    return df


def run_anova(df, value_col="normalized_distance", group_col="cluster_id"):
    groups = [
        g[value_col].dropna().values
        for _, g in df.groupby(group_col)
        if g[value_col].notna().sum() > 1
    ]
    if len(groups) > 1:
        f_val, p_val = f_oneway(*groups)
        return float(f_val), float(p_val)
    return np.nan, np.nan

In [ ]:
patient_dirs = sorted(
    os.path.join(RESULTS_ROOT, d) for d in os.listdir(RESULTS_ROOT)
    if os.path.isdir(os.path.join(RESULTS_ROOT, d))
)

anova_summaries = {}

for region in NORM_REGIONS:
    print(f"\n{'='*60}\nRegion: {region}")
    global_majority_cluster = find_global_majority_cluster(patient_dirs, region)
    if global_majority_cluster is None:
        print(f"  No usable data for region {region} -- skip")
        continue
    print(f"  Global most-populated cluster: {global_majority_cluster}")

    anova_rows = []
    for pdir in patient_dirs:
        patient_id = os.path.basename(pdir)
        csv_path = os.path.join(pdir, f"{region}_clusterwise_cosine_distances.csv")
        if not os.path.exists(csv_path):
            continue

        df_norm = normalize_patient_region(csv_path, global_majority_cluster, region)
        if df_norm is None:
            continue

        out_path = os.path.join(pdir, f"{region}_clusterwise_cosine_distances_normalized.csv")
        df_norm.to_csv(out_path, index=False)

        f_val, p_val = run_anova(df_norm)
        anova_rows.append({
            "patient": patient_id, "region": region,
            "reference_cluster": global_majority_cluster,
            "noise_floor": df_norm["noise_floor"].iloc[0],
            "f_val": f_val, "p_val": p_val,
        })

    if anova_rows:
        anova_df = pd.DataFrame(anova_rows)
        anova_path = os.path.join(RESULTS_ROOT, f"{region}_anova_summary_global_most_populated.csv")
        anova_df.to_csv(anova_path, index=False)
        anova_summaries[region] = anova_df
        print(f"  Saved ANOVA summary: {anova_path}")

print("\nDone.")

## Results

In [ ]:
for region, df in anova_summaries.items():
    print(f"--- {region} ---")
    display(df[["patient", "noise_floor", "f_val", "p_val"]])